# RF-DETR Predictions Visualization — dataset2_split test

For each sequence: 2 frames × 4 columns (raw, GT, dataset2_augs preds, correct_single_label preds)

In [ ]:
import re, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as patches
from pathlib import Path
from PIL import Image
from collections import defaultdict
from rfdetr import RFDETRSmall

ROOT = Path('/home/dsa/stenosis')
IMG_DIR = ROOT / 'data' / 'dataset2_split' / 'test' / 'images'
LBL_DIR = ROOT / 'data' / 'dataset2_split' / 'test' / 'labels'

FRAMES_PER_SEQ = 2
CONF = 0.15

In [ ]:
# Group images by sequence
seq_re = re.compile(r'^(\d+_\d+_\d+)_\d+_bmp_jpg')
sequences = defaultdict(list)
for p in sorted(IMG_DIR.glob('*.jpg')):
    m = seq_re.match(p.name)
    if m:
        sequences[m.group(1)].append(p)

# Pick evenly spaced frames
selected = []
for sid in sorted(sequences):
    imgs = sequences[sid]
    n = len(imgs)
    idx = np.linspace(0, n - 1, min(FRAMES_PER_SEQ, n), dtype=int)
    selected.extend(imgs[i] for i in idx)

print(f'{len(sequences)} sequences, {len(selected)} frames selected')

In [ ]:
# Load models — pretrain_weights handles checkpoint loading + num_classes + device
model_ds2 = RFDETRSmall(
    pretrain_weights=str(ROOT / 'rfdetr_runs' / 'dataset2_augs' / 'checkpoint_best_total.pth'),
    device='cuda:0',
)
model_csl = RFDETRSmall(
    pretrain_weights=str(ROOT / 'rfdetr_runs' / 'correct_single_labelf_rfdetr_arcade_dataset2' / 'checkpoint_best_total.pth'),
    device='cuda:0',
)
print('Models loaded')

In [ ]:
# Run inference
preds_ds2, preds_csl = {}, {}
for i, p in enumerate(selected):
    preds_ds2[p.name] = model_ds2.predict(str(p), threshold=CONF)
    preds_csl[p.name] = model_csl.predict(str(p), threshold=CONF)
    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(selected)}')
print(f'Done — {len(selected)} frames')

In [ ]:
def read_yolo_boxes(lbl_path, w, h):
    boxes = []
    if not lbl_path.exists() or lbl_path.stat().st_size == 0:
        return boxes
    for line in lbl_path.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 5: continue
        cx, cy, bw, bh = map(float, parts[1:5])
        boxes.append(((cx - bw/2)*w, (cy - bh/2)*h, (cx + bw/2)*w, (cy + bh/2)*h))
    return boxes

def draw_rects(ax, boxes, color, lw=2, show_conf=False):
    for b in boxes:
        if show_conf:
            x1, y1, x2, y2, conf = b
        else:
            x1, y1, x2, y2 = b
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, lw=lw, ec=color, fc='none'))
        if show_conf:
            ax.text(x1, y1-3, f'{conf:.2f}', color=color, fontsize=7,
                    bbox=dict(boxstyle='round,pad=0.1', fc='black', alpha=0.6))

def det_to_boxes(det, thresh):
    if det is None or len(det) == 0:
        return []
    mask = det.confidence >= thresh
    return [(x1,y1,x2,y2,c) for (x1,y1,x2,y2),c in zip(det.xyxy[mask], det.confidence[mask])]

In [ ]:
for p in selected:
    img = np.array(Image.open(p))
    h, w = img.shape[:2]
    gt = read_yolo_boxes(LBL_DIR / (p.stem + '.txt'), w, h)
    ds2 = det_to_boxes(preds_ds2[p.name], CONF)
    csl = det_to_boxes(preds_csl[p.name], CONF)

    fig, ax = plt.subplots(1, 4, figsize=(24, 6))
    titles = ['Raw', f'GT ({len(gt)})', f'dataset2_augs ({len(ds2)})', f'correct_single_label ({len(csl)})']
    for a, t in zip(ax, titles):
        a.imshow(img); a.set_title(t, fontsize=11); a.axis('off')

    draw_rects(ax[1], gt, 'lime')
    draw_rects(ax[2], gt, 'lime', lw=1); draw_rects(ax[2], ds2, 'red', show_conf=True)
    draw_rects(ax[3], gt, 'lime', lw=1); draw_rects(ax[3], csl, 'cyan', show_conf=True)

    m = seq_re.match(p.name)
    fig.suptitle(f'{m.group(1)} — {p.name}', fontsize=12, y=1.02)
    plt.tight_layout(); plt.show()